Clima Campeonato Brasileiro e Premier League

In [1]:
import pandas as pd
import requests
import time

def processar_campeonatos():
    # Lista com as configurações de entrada e saída para cada campeonato
    arquivos_para_processar = [
        {
            "entrada": "Campeonato Premier League 2019 até 2024.csv",
            "saida": "Campeonato_Premier_League_Com_Clima.csv",
            "nome": "Premier League"
        },
        {
            "entrada": "Campeonato Brasileiro Série A 2020 até 2025.csv",
            "saida": "Campeonato_Brasileiro_Com_Clima.csv",
            "nome": "Brasileirão"
        }
    ]

    # Dicionário unificado de correções para ajudar a API a encontrar a localização exata
    correcoes = {
        "Newcastle": "Newcastle upon Tyne",
        "Everton": "Liverpool",
        "Athletico-PR": "Curitiba",
        "Bahia": "Salvador",
        "Flamengo": "Rio de Janeiro",
        "Red Bull Bragantino": "Bragança Paulista",
        "Santos": "Santos",
        "São Lourenço da Mata": "São Lourenço da Mata"
    }

    # Dicionário global que guardará o clima das cidades. 
    # Fica fora do loop para reaproveitar dados caso a mesma cidade apareça mais de uma vez
    clima_por_cidade = {}
    limite_historico = pd.Timestamp.today() - pd.Timedelta(days=5)

    # Função interna para buscar a coordenada tentando UK e depois Brasil
    def buscar_coordenada(cidade_nome):
        urls_tentativas = [
            f"https://geocoding-api.open-meteo.com/v1/search?name={cidade_nome}, United Kingdom&count=1&language=en",
            f"https://geocoding-api.open-meteo.com/v1/search?name={cidade_nome}, Brasil&count=1&language=pt"
        ]
        for url in urls_tentativas:
            try:
                r = requests.get(url)
                r.raise_for_status()
                res = r.json()
                if 'results' in res and len(res['results']) > 0:
                    return res['results'][0]['latitude'], res['results'][0]['longitude']
            except Exception:
                pass
        return None, None

    # Loop principal que processa um campeonato por vez
    for liga in arquivos_para_processar:
        print(f"\n{'='*50}")
        print(f" Iniciando processamento: {liga['nome']}")
        print(f"{'='*50}")
        
        try:
            df = pd.read_csv(liga['entrada'])
        except FileNotFoundError:
            print(f"-> ERRO: Arquivo '{liga['entrada']}' não encontrado. Pulando para o próximo...")
            continue

        df['Data_Hora_DT'] = pd.to_datetime(df['Data_Hora'], format='%d.%m.%Y %H:%M', errors='coerce')
        
        if 'Estadio_Local' in df.columns:
            df['Cidade_Busca'] = df['Estadio_Local'].str.extract(r'\((.*?)\)')[0]
            df['Cidade_Busca'] = df['Cidade_Busca'].fillna(df['Mandante'])
        else:
            df['Cidade_Busca'] = df['Mandante']
            
        cidades_unicas = df['Cidade_Busca'].dropna().unique()
        print(f"Encontradas {len(cidades_unicas)} cidades/locais únicos para o {liga['nome']}.")

        for cidade in cidades_unicas:
            # Pula a requisição se já baixamos essa cidade nesta execução
            if cidade in clima_por_cidade:
                continue 

            nome_pesquisa = correcoes.get(cidade, cidade)
            lat, lon = buscar_coordenada(nome_pesquisa)
            
            if lat is not None and lon is not None:
                df_cidade = df[(df['Cidade_Busca'] == cidade) & df['Data_Hora_DT'].notna()]
                if df_cidade.empty:
                    continue
                    
                data_minima = df_cidade['Data_Hora_DT'].min() - pd.Timedelta(days=1)
                data_maxima = df_cidade['Data_Hora_DT'].max() + pd.Timedelta(days=1)
                
                if data_maxima > limite_historico:
                    data_maxima = limite_historico
                    
                min_str = data_minima.strftime('%Y-%m-%d')
                max_str = data_maxima.strftime('%Y-%m-%d')
                
                print(f"Baixando histórico de {min_str} a {max_str} para: {cidade}...")
                weather_url = (
                    f"https://archive-api.open-meteo.com/v1/archive?"
                    f"latitude={lat}&longitude={lon}&start_date={min_str}&end_date={max_str}"
                    f"&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m&timezone=auto"
                )
                
                sucesso = False
                for tentativa in range(3):
                    w_r = requests.get(weather_url)
                    
                    if w_r.status_code == 429:
                        print(f"   -> Limite da API atingido. Pausando 15s... (Tentativa {tentativa+1}/3)")
                        time.sleep(15)
                        continue
                        
                    if w_r.status_code == 200:
                        w_data = w_r.json()
                        if 'hourly' in w_data:
                            df_weather = pd.DataFrame(w_data['hourly'])
                            df_weather['time'] = pd.to_datetime(df_weather['time'])
                            clima_por_cidade[cidade] = df_weather
                            sucesso = True
                            break
                
                if not sucesso:
                    print(f"-> Falha contínua ao baixar dados de '{cidade}'.")
            else:
                print(f"-> Atenção: Coordenadas não encontradas (BR ou UK) para '{cidade}'.")
                
            time.sleep(3) # Pausa respeitosa para a Open-Meteo

        print(f"\nCruzando os dados climáticos com as partidas do {liga['nome']}...")
        df['Data_Hora_Arredondada'] = df['Data_Hora_DT'].dt.round('h')
        
        temperaturas, umidades, ventos = [], [], []
        
        for _, row in df.iterrows():
            cidade = row['Cidade_Busca']
            hora_jogo = row['Data_Hora_Arredondada']
            t, u, v = None, None, None
            
            if pd.notna(hora_jogo) and pd.notna(cidade) and cidade in clima_por_cidade:
                df_clima = clima_por_cidade[cidade]
                match = df_clima[df_clima['time'] == hora_jogo]
                if not match.empty:
                    t = match.iloc[0]['temperature_2m']
                    u = match.iloc[0]['relative_humidity_2m']
                    v = match.iloc[0]['wind_speed_10m']
                    
            temperaturas.append(t)
            umidades.append(u)
            ventos.append(v)
            
        df['Temperatura_C'] = temperaturas
        df['Umidade_Relativa_%'] = umidades
        df['Velocidade_Vento_kmh'] = ventos
        
        # Limpa as colunas auxiliares
        df = df.drop(columns=['Data_Hora_DT', 'Cidade_Busca', 'Data_Hora_Arredondada'])
        
        # Salva o arquivo CSV com o nome da respectiva liga
        df.to_csv(liga['saida'], index=False)
        print(f"Concluído! O arquivo '{liga['saida']}' foi salvo com sucesso.")

if __name__ == "__main__":
    processar_campeonatos()


 Iniciando processamento: Premier League
Encontradas 21 cidades/locais únicos para o Premier League.
Baixando histórico de 2019-08-10 a 2026-03-16 para: Manchester...
Baixando histórico de 2019-08-10 a 2025-05-19 para: Leicester...
Baixando histórico de 2019-08-10 a 2026-03-16 para: Newcastle...
-> Atenção: Coordenadas não encontradas (BR ou UK) para 'Londres'.
Baixando histórico de 2019-08-09 a 2026-03-16 para: Bournemouth...
Baixando histórico de 2019-08-09 a 2026-03-16 para: Burnley...
Baixando histórico de 2019-08-09 a 2022-05-16 para: Watford...
Baixando histórico de 2019-08-08 a 2026-03-16 para: Liverpool...
Baixando histórico de 2019-08-18 a 2026-03-04 para: Wolverhampton...
Baixando histórico de 2019-08-17 a 2024-05-20 para: Sheffield...
Baixando histórico de 2019-08-16 a 2026-03-16 para: Birmingham...
Baixando histórico de 2019-08-16 a 2026-03-16 para: Brighton...
Baixando histórico de 2019-08-16 a 2022-05-23 para: Norwich...
Baixando histórico de 2019-08-16 a 2025-05-26 para